# Deep Learning for Image Inpainting

Image inpainting reconstructs missing regions of an image using the visible surrounding content. This project will compare a lightweight **U-Net**, a lightweight **Vision Transformer (ViT)**, and their **weighted ensemble**.

## Phase 1: Dataset preparation

This phase loads original images, creates reproducible data splits, and checks the data. Run the cells in order in Google Colab. Masking and models will be added in later phases.

In [ ]:
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

SEED = 42
IMAGE_SIZE = 128
BATCH_SIZE = 32
NUM_WORKERS = 2
TRAIN_FRACTION = 0.8
# Download into Colab's temporary runtime storage when this cell is run there.
DATA_ROOT = Path('/content/data') if Path('/content').is_dir() else Path('data')

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'Dataset location: {DATA_ROOT.resolve()}')

## Oxford-IIIT Pet Dataset and splits

Oxford-IIIT Pet contains photographs of cats and dogs from 37 pet categories. We use the photographs for reconstruction; category labels returned by torchvision are not needed for inpainting.

The **official test split remains untouched** and reserved for final evaluation. We split only the official **trainval** split into **80% training / 20% validation**, using a fixed seed. Both future models will use these same subsets.

Images are loaded as RGB by torchvision, resized to **128 × 128**, and converted to floating-point tensors in **[0, 1]**. No ImageNet mean/std normalization is applied. The first run downloads the dataset into the runtime's data folder and may take a few minutes.

In [ ]:
image_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
])

trainval_dataset = datasets.OxfordIIITPet(
    root=DATA_ROOT,
    split='trainval',
    target_types='category',
    transform=image_transform,
    download=True,
)
test_dataset = datasets.OxfordIIITPet(
    root=DATA_ROOT,
    split='test',
    target_types='category',
    transform=image_transform,
    download=True,
)

train_size = int(TRAIN_FRACTION * len(trainval_dataset))
val_size = len(trainval_dataset) - train_size
train_dataset, val_dataset = random_split(
    trainval_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED),
)

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    generator=torch.Generator().manual_seed(SEED),
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
)

In [ ]:
print(f'Official trainval: {len(trainval_dataset):,}')
print(f'Training:         {len(train_dataset):,}')
print(f'Validation:       {len(val_dataset):,}')
print(f'Official test:     {len(test_dataset):,}')

# Check that training and validation partition trainval without overlap.
train_indices = set(train_dataset.indices)
val_indices = set(val_dataset.indices)
assert len(train_indices) == train_size > 0
assert len(val_indices) == val_size > 0
assert train_indices.isdisjoint(val_indices)
assert train_indices | val_indices == set(range(len(trainval_dataset)))
assert test_dataset is not trainval_dataset
assert len(test_dataset) > 0

# Verify that the same seed reproduces exactly the same split.
expected_indices = torch.randperm(
    len(trainval_dataset), generator=torch.Generator().manual_seed(SEED)
).tolist()
assert train_dataset.indices == expected_indices[:train_size]
assert val_dataset.indices == expected_indices[train_size:]

# Inspect a training batch only; do not iterate over the test set.
images, _ = next(iter(train_loader))
print(f'Batch shape: {list(images.shape)}')
print(f'Pixel range: [{images.min().item():.3f}, {images.max().item():.3f}]')
assert images.shape == (min(BATCH_SIZE, train_size), 3, IMAGE_SIZE, IMAGE_SIZE)
assert images.dtype == torch.float32
assert torch.isfinite(images).all().item()
assert 0.0 <= images.min().item() <= images.max().item() <= 1.0
print('All split and training-batch sanity checks passed.')

In [ ]:
# Display eight original training images from the checked batch.
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for index, ax in enumerate(axes.flat):
    ax.imshow(images[index].permute(1, 2, 0).numpy())
    ax.set_title(f'Training image {index + 1}')
    ax.axis('off')
fig.suptitle('Original training images (128 x 128)')
plt.tight_layout()
plt.show()

## Phase 2: Artificial rectangular masks

The **masked image is the model input**, and the **unchanged original image is the reconstruction target**. We use central or randomly positioned rectangular masks, with examples of sizes **24 x 24**, **32 x 32**, and **48 x 48**.

For each RGB image of shape `[3, H, W]`, the binary mask has shape `[1, H, W]`: **1 (True) marks missing pixels**, and **0 (False) marks visible pixels**. The same mask covers all three color channels. Missing pixels are filled with **0 (black)** in a new tensor, leaving the target unchanged. In the mask plots, white indicates the missing region.

Supplying a seed reproduces the random position for the same image dimensions and mask size. Use different seeds for different examples. These examples only verify masking; the training-mask mixture will be decided later.

In [ ]:
MASK_SIZES = [(24, 24), (32, 32), (48, 48)]
MASK_FILL_VALUE = 0.0


def make_rectangular_mask(image, mask_size=(32, 32), position='central', seed=None):
    """Return a boolean [1, H, W] mask: True means a missing pixel."""
    if image.ndim != 3 or image.shape[0] != 3:
        raise ValueError('Expected an RGB image with shape [3, H, W].')
    height, width = image.shape[-2:]
    mask_height, mask_width = mask_size
    if not (0 < mask_height <= height and 0 < mask_width <= width):
        raise ValueError('Mask dimensions must be positive and fit inside the image.')

    if position == 'central':
        top = (height - mask_height) // 2
        left = (width - mask_width) // 2
    elif position == 'random':
        # A local seeded generator does not reset the global random state.
        generator = None if seed is None else torch.Generator().manual_seed(seed)
        top = torch.randint(height - mask_height + 1, (1,), generator=generator).item()
        left = torch.randint(width - mask_width + 1, (1,), generator=generator).item()
    else:
        raise ValueError("position must be 'central' or 'random'.")

    mask = torch.zeros((1, height, width), dtype=torch.bool, device=image.device)
    mask[:, top:top + mask_height, left:left + mask_width] = True
    return mask


def apply_mask(image, mask):
    """Fill the hole in a new tensor; preserve the original image."""
    return image.masked_fill(mask, MASK_FILL_VALUE)

In [ ]:
# Use only training images from Phase 1; the test set stays untouched.
originals_before_masking = images.clone()
for position in ('central', 'random'):
    for mask_size in MASK_SIZES:
        target = images[0]
        mask = make_rectangular_mask(target, mask_size, position, seed=SEED)
        masked_image = apply_mask(target, mask)

        assert mask.shape == (1, IMAGE_SIZE, IMAGE_SIZE)
        assert mask.dtype == torch.bool
        assert masked_image.shape == target.shape
        # Count spatial pixels once, not once per RGB channel.
        assert mask.sum().item() == mask_size[0] * mask_size[1]
        missing_channels = mask.expand_as(target)
        assert torch.all(masked_image[missing_channels] == MASK_FILL_VALUE).item()
        assert torch.equal(masked_image[~missing_channels], target[~missing_channels])
        assert torch.equal(images, originals_before_masking)

        if position == 'central':
            top = (IMAGE_SIZE - mask_size[0]) // 2
            left = (IMAGE_SIZE - mask_size[1]) // 2
            assert mask[:, top:top + mask_size[0], left:left + mask_size[1]].all().item()
        else:
            repeated_mask = make_rectangular_mask(target, mask_size, position, seed=SEED)
            assert torch.equal(mask, repeated_mask)

        print(f'{position:7s} {mask_size}: {mask.sum().item()} missing pixels - passed')

print('All Phase 2 sanity checks passed; original targets are unchanged.')

In [ ]:
# Two figures, each with three training images and all three mask sizes.
# Reuse the same originals across figures for a direct position comparison.
for position in ('central', 'random'):
    fig, axes = plt.subplots(3, 3, figsize=(9, 9))
    for row, mask_size in enumerate(MASK_SIZES):
        target = images[row]
        mask = make_rectangular_mask(target, mask_size, position, seed=SEED + row)
        masked_image = apply_mask(target, mask)

        axes[row, 0].imshow(target.permute(1, 2, 0).cpu().numpy())
        axes[row, 1].imshow(masked_image.permute(1, 2, 0).cpu().numpy())
        axes[row, 2].imshow(mask[0].cpu().numpy(), cmap='gray', vmin=0, vmax=1)
        axes[row, 0].set_title(f'Original (training image {row + 1})')
        axes[row, 1].set_title(f'Masked ({mask_size[0]} x {mask_size[1]})')
        axes[row, 2].set_title('Mask (white = missing)')
        for ax in axes[row]:
            ax.axis('off')

    fig.suptitle(f'{position.capitalize()} rectangular masks')
    plt.tight_layout()
    plt.show()

assert torch.equal(images, originals_before_masking)

## Phase 3: Lightweight U-Net (untrained)

U-Net reconstructs an image through an **encoder**, a **bottleneck**, and a **decoder**:

- The encoder extracts features at 128 x 128, 64 x 64, and 32 x 32 resolutions, with 32, 64, and 128 channels. Max pooling halves the spatial size between stages.
- The bottleneck uses 256 channels at 16 x 16 to represent the surrounding image context.
- The decoder upsamples back to 128 x 128 using transposed convolutions and refines features with convolution blocks.
- **Skip connections** concatenate encoder features with decoder features at matching resolutions. They give the decoder direct access to spatial details, such as edges and textures in the visible region, which helps image reconstruction and inpainting.

Each convolution block contains two 3 x 3 convolutions (padding 1), each followed by ReLU. Decoder upsampling uses 2 x 2 transposed convolutions with stride 2. A final 3 x 3 convolution maps 32 channels to 3 RGB channels; sigmoid constrains predictions to [0, 1].

The model takes only the masked RGB image and predicts a full RGB image of the same size. It starts with random weights: this section checks shapes and value ranges only. **Untrained predictions are not experimental reconstruction results.**

In [ ]:
from torch import nn


def conv_block(in_channels, out_channels):
    """Two convolutions that preserve spatial dimensions."""
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
        nn.ReLU(),
    )


class UNet(nn.Module):
    """Small U-Net for [B, 3, 128, 128] masked RGB images."""

    def __init__(self):
        super().__init__()
        self.encoder1 = conv_block(3, 32)
        self.encoder2 = conv_block(32, 64)
        self.encoder3 = conv_block(64, 128)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.bottleneck = conv_block(128, 256)

        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.decoder3 = conv_block(128 + 128, 128)
        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.decoder2 = conv_block(64 + 64, 64)
        self.up1 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.decoder1 = conv_block(32 + 32, 32)

        self.output_conv = nn.Conv2d(32, 3, kernel_size=3, padding=1)
        self.output_activation = nn.Sigmoid()

    def forward(self, x):
        enc1 = self.encoder1(x)                  # [B, 32, 128, 128]
        enc2 = self.encoder2(self.pool(enc1))    # [B, 64, 64, 64]
        enc3 = self.encoder3(self.pool(enc2))    # [B, 128, 32, 32]
        bottleneck = self.bottleneck(self.pool(enc3))  # [B, 256, 16, 16]

        # Concatenate along the channel dimension at matching resolutions.
        dec3 = self.decoder3(torch.cat([self.up3(bottleneck), enc3], dim=1))
        dec2 = self.decoder2(torch.cat([self.up2(dec3), enc2], dim=1))
        dec1 = self.decoder1(torch.cat([self.up1(dec2), enc1], dim=1))
        return self.output_activation(self.output_conv(dec1))

In [ ]:
# Reuse the device detected in Phase 1 under the requested uppercase name.
DEVICE = device
# Make initialization reproducible when this cell is rerun.
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

unet = UNet().to(DEVICE)
trainable_parameters = sum(p.numel() for p in unet.parameters() if p.requires_grad)
print(f'U-Net device: {DEVICE}')
print(f'Trainable parameters: {trainable_parameters:,}')

In [ ]:
# Small forward-pass example using training images and Phase 2 mask functions.
# These two masks are examples, not a choice of training-mask mixture.
unet_targets = images[:2].clone()
unet_example_masks = torch.stack([
    make_rectangular_mask(unet_targets[0], (32, 32), position='central'),
    make_rectangular_mask(unet_targets[1], (32, 32), position='random', seed=SEED),
])
unet_inputs = torch.stack([
    apply_mask(target, mask)
    for target, mask in zip(unet_targets, unet_example_masks)
]).to(DEVICE)

unet.eval()
with torch.no_grad():
    untrained_predictions = unet(unet_inputs)

assert unet_inputs.shape == (2, 3, IMAGE_SIZE, IMAGE_SIZE)
assert untrained_predictions.shape == unet_inputs.shape
assert torch.isfinite(untrained_predictions).all().item()
assert 0.0 <= untrained_predictions.min().item() <= untrained_predictions.max().item() <= 1.0
assert torch.equal(unet_targets, images[:2])

print(f'Input shape:  {list(unet_inputs.shape)}')
print(f'Output shape: {list(untrained_predictions.shape)}')
print(f'Untrained output range: [{untrained_predictions.min().item():.4f}, '
      f'{untrained_predictions.max().item():.4f}]')
print('Phase 3 forward-pass checks passed. No training has been performed.')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(9, 6))
for row in range(2):
    axes[row, 0].imshow(unet_targets[row].permute(1, 2, 0).cpu().numpy())
    axes[row, 1].imshow(unet_inputs[row].permute(1, 2, 0).cpu().numpy())
    axes[row, 2].imshow(untrained_predictions[row].permute(1, 2, 0).cpu().numpy())
    for ax, title in zip(axes[row], ('Original target', 'Masked input', 'UNTRAINED prediction')):
        ax.set_title(title)
        ax.axis('off')
fig.suptitle('Randomly initialized U-Net: sanity check only, not experimental results')
plt.tight_layout()
plt.show()